# Seasonal GVI Estimation — Gemini Batch API Inference

Estimates **spring, fall, and winter GVI** from summer street-view imagery using the Gemini Batch API (zero-shot).  
This notebook implements the selected configuration from the paper:  
**Gemini 2.5 Flash + Structured CoT + GVI+IMG+LOC+VEG** (paper Table 3 & 4).

**Pipeline**
1. Load summer metadata CSV (output of `02_segmentation_gvi.ipynb`)
2. Upload images to the Gemini File API
3. Build and submit a Gemini Batch job
4. Poll until the job completes
5. Parse results and save a per-image seasonal GVI CSV
6. Aggregate to point-level GVI and save `point_result.csv`

> Paper: *Seasonal GVI Estimation Without Seasonal Imagery: A Zero-Shot MLLM Framework for Urban Greenery Assessment* (SIGSPATIAL '26)  
> Code: https://github.com/sjooha111/summer-to-seasonal-GVI

In [ ]:
%pip install -q -U "google-genai>=1.34.0" pandas

## Step 0 — Configuration

**Edit only this cell** before running the notebook.  
All other cells can be run as-is.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
GEMINI_API_KEY = ""              # Gemini API key; leave blank to use GOOGLE_API_KEY env var
INPUT_CSV      = "../data/sample/metadata_sample.csv"
IMAGE_DIR      = ""              # Root dir containing summer images
                                 # (image_path in CSV is treated as absolute, or relative to IMAGE_DIR)
OUTPUT_DIR     = "../results"
OUTPUT_CSV     = "seasonal_gvi_results.csv"

# ── Model & prompt ──────────────────────────────────────────────────────────
MODEL           = "gemini-2.5-flash"
PROMPT_STRATEGY = "structured_cot"   # "direct" | "base_cot" | "structured_cot"

# ── Input components (paper Table 3) ───────────────────────────────────────
# GVI is always included as the primary numerical anchor (paper Section 4.1).
USE_IMAGE = True   # Include summer street-view image
USE_LOC   = True   # Include city / country / coordinates
USE_VEG   = True   # Include vegetation class composition

# ── Season target months — Northern Hemisphere default (paper Section 3.5) ──
SPRING_MONTH = 4    # April
FALL_MONTH   = 11   # November
WINTER_MONTH = 2    # February
# Southern Hemisphere override (auto-detected when country == 'Australia')
SH_SPRING_MONTH = 9
SH_FALL_MONTH   = 4
SH_WINTER_MONTH = 7

# ── Filtering (paper Section 3.2) ───────────────────────────────────────────
POINT_GVI_THRESHOLD = 0.01   # Exclude points whose summer GVI < this value

## Step 1 — Setup

In [ ]:
import json, time, os
from pathlib import Path
import pandas as pd
from google import genai
from google.genai import types

PROMPTS_DIR = Path("../prompts")
OUTPUT_PATH = Path(OUTPUT_DIR)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Load system prompt from the prompts/ directory
_prompt_file = {
    "direct":         "system_prompt_direct.txt",
    "base_cot":       "system_prompt_base_cot.txt",
    "structured_cot": "system_prompt_structured_cot.txt",
}[PROMPT_STRATEGY]
SYSTEM_PROMPT = (PROMPTS_DIR / _prompt_file).read_text(encoding="utf-8")

_api_key = GEMINI_API_KEY or os.environ.get("GOOGLE_API_KEY", "")
client   = genai.Client(api_key=_api_key)

print(f"Model          : {MODEL}")
print(f"Prompt strategy: {PROMPT_STRATEGY}")
print(f"Input flags    : image={USE_IMAGE}  loc={USE_LOC}  veg={USE_VEG}")

In [ ]:
# ── Response JSON schema ────────────────────────────────────────────────────
_base_schema = {
    "type": "object",
    "properties": {
        "gvi_spring": {"type": "number"},
        "gvi_fall":   {"type": "number"},
        "gvi_winter": {"type": "number"},
    },
    "required": ["gvi_spring", "gvi_fall", "gvi_winter"],
}
RESPONSE_SCHEMA = _base_schema.copy()
if PROMPT_STRATEGY != "direct":
    RESPONSE_SCHEMA["properties"]["rationale"] = {"type": "string"}
    RESPONSE_SCHEMA["required"] = ["gvi_spring", "gvi_fall", "gvi_winter", "rationale"]

MONTH_NAME = {
    1:"January", 2:"February", 3:"March",    4:"April",
    5:"May",     6:"June",     7:"July",      8:"August",
    9:"September",10:"October",11:"November",12:"December",
}


def build_user_prompt(row) -> str:
    is_south = str(row.get("country", "")).strip().lower() == "australia"
    months = {
        "spring": SH_SPRING_MONTH if is_south else SPRING_MONTH,
        "fall":   SH_FALL_MONTH   if is_south else FALL_MONTH,
        "winter": SH_WINTER_MONTH if is_south else WINTER_MONTH,
    }

    lines = []
    if USE_IMAGE and not (USE_LOC or USE_VEG):
        lines.append("[Summer Street View Image]")
    else:
        lines.append("[Summer Street View Image Metadata]")

    # GVI anchor — always included
    lines.append(f"- Summer GVI: {float(row['summer_gvi']) * 100:.2f}%")

    if USE_LOC:
        lines.append(
            f"- Location: {row['city']}, {row['country']}"
            f" (coordinates: {float(row['lat']):.4f}, {float(row['lon']):.4f})"
        )

    if USE_VEG:
        veg_cols = ["tree_ratio", "grass_ratio", "plant_ratio",
                    "palm_ratio", "flower_ratio", "field_ratio"]
        veg_lines = [
            f"  - {col.replace('_ratio', '')}: {float(row.get(col, 0))*100:.2f}%"
            for col in veg_cols if float(row.get(col, 0)) > 0
        ]
        lines.append("- Vegetation class composition:")
        lines.append("\n".join(veg_lines) if veg_lines else "  - (none detected)")

    lines += ["",
              "Predict GVI for:",
              f"(a) {MONTH_NAME[months['spring']]}",
              f"(b) {MONTH_NAME[months['fall']]}",
              f"(c) {MONTH_NAME[months['winter']]}"]
    return "\n".join(lines)

## Step 2 — Load & Filter Data

Loads the metadata CSV produced by `02_segmentation_gvi.ipynb`.  
Points with summer GVI below `POINT_GVI_THRESHOLD` are excluded — they have no visible vegetation  
and are assumed to remain at GVI = 0 across all seasons (paper Section 3.2).  
Individual images within valid points that have GVI = 0 are auto-assigned 0 without model inference.

In [ ]:
df = pd.read_csv(INPUT_CSV)
print(f"Loaded: {len(df)} rows")

n_before = len(df)
df = df[df["summer_gvi"] >= POINT_GVI_THRESHOLD].reset_index(drop=True)
print(f"After low-vegetation filter (>={POINT_GVI_THRESHOLD}): {len(df)} rows  (excluded {n_before - len(df)})")

df_infer = df[df["summer_gvi"] > 0].reset_index(drop=True)
print(f"Images sent to model: {len(df_infer)}")
df_infer.head(3)

## Step 3 — Upload Images to Gemini File API

Gemini Batch API requires images to be pre-uploaded via the File API and referenced by URI.  
A local cache file (`uri_cache.json`) is maintained so already-uploaded images are not re-uploaded on reruns.

> **Note:** Uploaded files expire after 48 hours. Re-run this cell if the cache contains stale URIs.

In [ ]:
def upload_images(df, client, output_dir: Path) -> dict:
    """Upload images to Gemini File API. Returns {abs_path: {uri, mime_type}}."""
    cache_path = output_dir / "uri_cache.json"
    uri_cache: dict = {}
    if cache_path.exists():
        with open(cache_path, encoding="utf-8") as f:
            uri_cache = json.load(f)
        print(f"Cache loaded: {len(uri_cache)} entries")

    image_dir = Path(IMAGE_DIR) if IMAGE_DIR else Path(".")
    pending = []
    for p in df["image_path"].unique():
        full = Path(p) if Path(p).is_absolute() else image_dir / p
        if str(full) not in uri_cache:
            pending.append(str(full))

    print(f"Uploading {len(pending)} new images ...")
    for i, fpath in enumerate(pending):
        uploaded = client.files.upload(
            file=fpath,
            config=types.UploadFileConfig(display_name=Path(fpath).name),
        )
        uri_cache[fpath] = {"uri": uploaded.uri, "mime_type": uploaded.mime_type}
        if (i + 1) % 20 == 0 or (i + 1) == len(pending):
            print(f"  {i+1}/{len(pending)}")
            with open(cache_path, "w", encoding="utf-8") as f:
                json.dump(uri_cache, f, ensure_ascii=False, indent=2)

    print(f"URI cache saved -> {cache_path}")
    return uri_cache


uri_cache = upload_images(df_infer, client, OUTPUT_PATH)

## Step 4 — Build Batch JSONL

Constructs one request per image in the Gemini Batch JSONL format.  
`thinking_budget=0` disables the model's internal thinking mode for a fair comparison  
with the Direct prompt strategy (paper Section 3.6).

In [ ]:
BATCH_PARAMS = dict(
    temperature=0,
    seed=42,
    max_output_tokens=4096,
    thinking_budget=0,   # 0 = disable thinking mode (paper Section 3.6)
)


def build_batch_jsonl(df, uri_cache: dict) -> list:
    image_dir = Path(IMAGE_DIR) if IMAGE_DIR else Path(".")
    lines = []
    for _, row in df.iterrows():
        key       = str(row["point_id"])
        user_text = build_user_prompt(row)

        gen_config = {
            "temperature":        BATCH_PARAMS["temperature"],
            "seed":               BATCH_PARAMS["seed"],
            "max_output_tokens":  BATCH_PARAMS["max_output_tokens"],
            "response_mime_type": "application/json",
            "response_schema":    RESPONSE_SCHEMA,
            "thinking_config":    {"thinking_budget": BATCH_PARAMS["thinking_budget"]},
        }

        if USE_IMAGE:
            full = Path(row["image_path"])
            if not full.is_absolute():
                full = image_dir / full
            cached = uri_cache[str(full)]
            parts = [
                {"file_data": {"file_uri": cached["uri"], "mime_type": cached["mime_type"]}},
                {"text": user_text},
            ]
        else:
            parts = [{"text": user_text}]

        request = {
            "system_instruction": {"parts": [{"text": SYSTEM_PROMPT}]},
            "contents": [{"role": "user", "parts": parts}],
            "generation_config": gen_config,
        }
        lines.append(json.dumps({"key": key, "request": request}, ensure_ascii=False))

    print(f"Built {len(lines)} requests")
    return lines


jsonl_lines = build_batch_jsonl(df_infer, uri_cache)

## Step 5 — Submit Batch Job

In [ ]:
def submit_batch(client, jsonl_lines: list, output_dir: Path):
    """Save JSONL, upload to File API, and submit batch job. Returns job object."""
    tag        = f"seasonal-gvi-{time.strftime('%Y%m%d-%H%M%S')}"
    jsonl_path = output_dir / f"batch_input_{tag}.jsonl"
    jsonl_path.write_text("\n".join(jsonl_lines), encoding="utf-8")
    print(f"JSONL saved -> {jsonl_path}  ({len(jsonl_lines)} requests)")

    uploaded = client.files.upload(
        file=str(jsonl_path),
        config=types.UploadFileConfig(display_name=tag, mime_type="application/jsonl"),
    )
    job = client.batches.create(
        model=MODEL, src=uploaded.name, config={"display_name": tag}
    )
    print(f"Batch submitted: {job.name}  state={job.state.name}")

    # Persist job name for recovery after kernel restart
    (output_dir / "last_batch_job.txt").write_text(job.name, encoding="utf-8")
    return job


batch_job = submit_batch(client, jsonl_lines, OUTPUT_PATH)
job_name  = batch_job.name
print(f"\nSave this for recovery: {job_name}")

## Step 6 — Poll Until Complete

Polls the job state every 60 seconds until it reaches a terminal state.  
Typical completion time: 10–30 minutes depending on dataset size.

In [ ]:
def poll_batch(client, job_name: str, interval: int = 60):
    terminal = {"JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED",
                "JOB_STATE_CANCELLED", "JOB_STATE_EXPIRED"}
    while True:
        job = client.batches.get(name=job_name)
        print(f"[{time.strftime('%H:%M:%S')}] {job.state.name}")
        if job.state.name in terminal:
            return job
        time.sleep(interval)


batch_job = poll_batch(client, job_name)
print(f"\nFinal state: {batch_job.state.name}")

## Step 7 — Parse Results & Save CSV

Parses the batch output and joins with the original metadata.  
Output columns: `gvi_spring`, `gvi_fall`, `gvi_winter` (0–100 scale, matching model output) and `rationale`.

In [ ]:
def parse_batch_output(client, job) -> pd.DataFrame:
    if job.state.name != "JOB_STATE_SUCCEEDED":
        raise RuntimeError(f"Job did not succeed: {job.state.name}")

    content = client.files.download(file=job.dest.file_name).decode("utf-8")
    rows = []
    for line in content.splitlines():
        if not line.strip():
            continue
        item = json.loads(line)
        key  = item["key"]

        if item.get("error"):
            rows.append({"point_id": key, "error": str(item["error"])})
            continue

        parts       = item["response"]["candidates"][0]["content"]["parts"]
        content_str = next(
            (p["text"] for p in reversed(parts) if not p.get("thought", False)),
            parts[-1]["text"],
        )

        try:
            parsed = json.loads(content_str)
        except json.JSONDecodeError as e:
            rows.append({"point_id": key, "error": f"json_parse_error: {e} | raw: {content_str[:200]}"})
            continue

        rows.append({
            "point_id":   key,
            "gvi_spring": parsed.get("gvi_spring"),
            "gvi_fall":   parsed.get("gvi_fall"),
            "gvi_winter": parsed.get("gvi_winter"),
            "rationale":  parsed.get("rationale"),
            "error":      None,
        })

    return pd.DataFrame(rows)


results_df = parse_batch_output(client, batch_job)
print(f"Parsed {len(results_df)} results  |  errors: {results_df['error'].notna().sum()}")

output_df = df_infer.merge(results_df, on="point_id", how="left")
out_path  = OUTPUT_PATH / OUTPUT_CSV
output_df.to_csv(out_path, index=False, encoding="utf-8")
print(f"Saved -> {out_path}")
output_df.head()

## Step 8 — Point-Level Aggregation

Combines inference results with the full point list (including LACK-filtered points):
- **USE points**: `gvi_spring/fall/winter` = model output ÷ 100 (converted to 0–1 scale)
- **LACK points**: `gvi_spring/fall/winter = NaN`, `lack = 'v'`  
  (structurally vegetation-free; assumed GVI = 0 year-round, paper Section 3.2)

Output: `point_result.csv`

In [ ]:
# Load full dataset (all points, including LACK-excluded)
df_all = pd.read_csv(INPUT_CSV)

# Load per-image inference output saved in Step 7
infer_out = pd.read_csv(OUTPUT_PATH / OUTPUT_CSV)

# One row per unique point (simple mode: point_id == image_id)
pt = df_all[['point_id', 'lat', 'lon', 'summer_gvi']].drop_duplicates('point_id').copy()

# Merge seasonal GVI from inference results
gvi_cols = ['point_id', 'gvi_spring', 'gvi_fall', 'gvi_winter']
pt = pt.merge(infer_out[gvi_cols], on='point_id', how='left')

# Scale model output from 0-100 → 0-1
for col in ['gvi_spring', 'gvi_fall', 'gvi_winter']:
    pt[col] = pt[col] / 100

# LACK flag: points excluded from inference remain NaN; mark them explicitly
pt['lack'] = ''
pt.loc[pt['summer_gvi'] < POINT_GVI_THRESHOLD, 'lack'] = 'v'

# Final column order
pt_result = pt[['point_id', 'lat', 'lon', 'summer_gvi',
                 'gvi_spring', 'gvi_fall', 'gvi_winter', 'lack']]

pt_path = OUTPUT_PATH / 'point_result.csv'
pt_result.to_csv(pt_path, index=False, encoding='utf-8')
print(f'Point result saved -> {pt_path}')
print(f'  Total points : {len(pt_result)}')
print(f'  USE (inferred): {(pt_result["lack"] == "").sum()}')
print(f'  LACK          : {(pt_result["lack"] == "v").sum()}')
pt_result.head()

---

## Resume — If the Kernel Was Restarted

If the kernel restarted after batch submission, run this cell to reload the job and skip to parsing.  
**Re-run Step 0 (CONFIG) and Step 1 (Setup) first**, then run the cell below.

In [ ]:
# ── Resume after kernel restart ─────────────────────────────────────────────
# Re-run Step 0 (CONFIG) and Step 1 (Setup) first, then run this cell.

OUTPUT_PATH = Path(OUTPUT_DIR)
_api_key    = GEMINI_API_KEY or os.environ.get("GOOGLE_API_KEY", "")
client      = genai.Client(api_key=_api_key)

job_name  = (OUTPUT_PATH / "last_batch_job.txt").read_text(encoding="utf-8").strip()
batch_job = client.batches.get(name=job_name)
print(f"Job: {job_name}  state={batch_job.state.name}")

if batch_job.state.name not in {"JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED",
                                 "JOB_STATE_CANCELLED", "JOB_STATE_EXPIRED"}:
    batch_job = poll_batch(client, job_name)

df       = pd.read_csv(INPUT_CSV)
df_infer = df[df["summer_gvi"] >= POINT_GVI_THRESHOLD].reset_index(drop=True)
df_infer = df_infer[df_infer["summer_gvi"] > 0].reset_index(drop=True)

results_df = parse_batch_output(client, batch_job)
output_df  = df_infer.merge(results_df, on="point_id", how="left")
output_df.to_csv(OUTPUT_PATH / OUTPUT_CSV, index=False, encoding="utf-8")
print(f"Saved -> {OUTPUT_PATH / OUTPUT_CSV}")